In [1]:
import pandas as pd
import os
from pandas.api.types import is_numeric_dtype

TRAIN_PATH = "../../data/train_processed.csv"
TEST_PATH  = "../../data/test_processed.csv"

df_tr = pd.read_csv(TRAIN_PATH).drop_duplicates().reset_index(drop=True)
df_te = pd.read_csv(TEST_PATH).drop_duplicates().reset_index(drop=True)

# target
y = df_tr["credit"]
X_tr = df_tr.drop(columns=["credit"])
X_te = df_te.copy()

# 컬럼 분리
num_cols = [c for c in X_tr.columns if is_numeric_dtype(X_tr[c])]
cat_cols = [c for c in X_tr.columns if c not in num_cols]

print("num:", num_cols)
print("cat:", cat_cols)

num: ['child_num', 'income_total', 'FLAG_MOBIL', 'work_phone', 'phone', 'email', 'family_size', 'begin_month', 'age', 'employment_years', 'income_per_person']
cat: ['gender', 'car', 'reality', 'income_type', 'edu_type', 'family_type', 'house_type', 'occyp_type']


In [2]:
for c in num_cols:
    med = X_tr[c].median()
    X_tr[c] = X_tr[c].fillna(med)
    X_te[c] = X_te[c].fillna(med)

In [3]:
for c in cat_cols:
    X_tr[c] = X_tr[c].astype(str).fillna("MISSING")
    X_te[c] = X_te[c].astype(str).fillna("MISSING")

In [4]:
os.makedirs("../../data/fe", exist_ok=True)

train_cat = pd.concat([X_tr, y], axis=1)

train_cat.to_csv("../../data/fe/train_cat_v2.csv", index=False)
X_te.to_csv("../../data/fe/test_cat_v2.csv", index=False)

print("✅ CatBoost v2 FE saved")

✅ CatBoost v2 FE saved


In [5]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold

# (중복 제거는 이미 했다면 스킵해도 됨)
df_tr = pd.read_csv("../../data/train_processed.csv").drop_duplicates().reset_index(drop=True)
df_te = pd.read_csv("../../data/test_processed.csv").drop_duplicates().reset_index(drop=True)

y = df_tr["credit"].astype(int)          # float -> int (0/1/2)
X = df_tr.drop(columns=["credit"]).copy()
X_test = df_te.copy()

num_cols = [c for c in X.columns if is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# 수치 결측: train 중앙값으로 채움(안정)
for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

# 범주형: 문자열화 + 결측 처리 (CatBoost 권장)
for c in cat_cols:
    X[c] = X[c].astype(str).fillna("MISSING")
    X_test[c] = X_test[c].astype(str).fillna("MISSING")

cat_features = [X.columns.get_loc(c) for c in cat_cols]
print("n_classes:", y.nunique(), "cat_cols:", len(cat_cols), "num_cols:", len(num_cols))

n_classes: 3 cat_cols: 8 num_cols: 11


In [6]:
class_counts = y.value_counts().sort_index().values
class_weights = (class_counts.sum() / (len(class_counts) * class_counts)).tolist()

In [7]:
from sklearn.metrics import accuracy_score

n_classes = y.nunique()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros((len(X), n_classes))
test_pred = np.zeros((len(X_test), n_classes))
models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    te_pool  = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=6000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=6,
        random_seed=42,
        verbose=200,
        early_stopping_rounds=300
        # ✅ class_weights 제거
    )

    model.fit(tr_pool, eval_set=val_pool, use_best_model=True)

    oof[val_idx] = model.predict_proba(X_val)
    test_pred += model.predict_proba(te_pool) / skf.n_splits
    models.append(model)

    val_pred_label = np.argmax(oof[val_idx], axis=1)
    acc = accuracy_score(y_val, val_pred_label)
    print(f"[Fold {fold}] best_iter={model.get_best_iteration()}  val_acc={acc:.5f}")

oof_label = np.argmax(oof, axis=1)
print("OOF accuracy:", accuracy_score(y, oof_label))

0:	learn: 1.0817741	test: 1.0817029	best: 1.0817029 (0)	total: 191ms	remaining: 19m 3s
200:	learn: 0.8014241	test: 0.8059094	best: 0.8059094 (200)	total: 8.41s	remaining: 4m 2s
400:	learn: 0.7779628	test: 0.7987983	best: 0.7987983 (400)	total: 17.9s	remaining: 4m 10s
600:	learn: 0.7471641	test: 0.7904216	best: 0.7904216 (600)	total: 27.9s	remaining: 4m 10s
800:	learn: 0.7216593	test: 0.7850692	best: 0.7850595 (798)	total: 37.7s	remaining: 4m 4s
1000:	learn: 0.6964702	test: 0.7802175	best: 0.7802175 (1000)	total: 47.3s	remaining: 3m 56s
1200:	learn: 0.6745638	test: 0.7774209	best: 0.7774076 (1194)	total: 56.9s	remaining: 3m 47s
1400:	learn: 0.6535552	test: 0.7737603	best: 0.7737588 (1399)	total: 1m 6s	remaining: 3m 38s
1600:	learn: 0.6323589	test: 0.7714738	best: 0.7714114 (1593)	total: 1m 16s	remaining: 3m 29s
1800:	learn: 0.6127139	test: 0.7698195	best: 0.7697742 (1795)	total: 1m 25s	remaining: 3m 20s
2000:	learn: 0.5935847	test: 0.7677165	best: 0.7676723 (1999)	total: 1m 35s	remainin

In [8]:
sub_label = pd.DataFrame({
    "credit": np.argmax(test_pred, axis=1)
})
sub_label.to_csv("../../data/submission_cat_v2_label.csv", index=False)
print("saved: ../../data/submission_cat_v2_label.csv")

saved: ../../data/submission_cat_v2_label.csv


In [9]:
sub_proba = pd.DataFrame(test_pred, columns=[f"credit_{i}" for i in range(n_classes)])
sub_proba.to_csv("../../data/submission_cat_v2_proba.csv", index=False)
print("saved: ../../data/submission_cat_v2_proba.csv")

saved: ../../data/submission_cat_v2_proba.csv


In [10]:
import pandas as pd
import numpy as np

TRAIN_PATH = "../../data/train_processed.csv"
TEST_PATH  = "../../data/test_processed.csv"

# 제출용 기준 길이(원본 train 길이 = 26457이어야 함)
df_tr_raw = pd.read_csv(TRAIN_PATH)
df_te_raw = pd.read_csv(TEST_PATH)

start_idx = len(df_tr_raw)  # 26457
submit_index = np.arange(start_idx, start_idx + len(df_te_raw))  # 26457~36456

print("start_idx:", start_idx)
print("test rows:", len(df_te_raw))
print("index range:", submit_index[0], submit_index[-1])

start_idx: 26457
test rows: 10000
index range: 26457 36456


In [11]:
class_counts = y.value_counts().sort_index().values
class_weights = (class_counts.sum() / (len(class_counts) * class_counts)).tolist()
print(class_weights)

[2.702723311546841, 1.3944247737874444, 0.5227770754319427]


In [12]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold

TRAIN_PATH = "../../data/train_processed.csv"
TEST_PATH  = "../../data/test_processed.csv"

# raw (제출 기준)
df_tr_raw = pd.read_csv(TRAIN_PATH)
df_te_raw = pd.read_csv(TEST_PATH)

# train은 dedup(선택) - 성능 위해 OK
df_tr = df_tr_raw.drop_duplicates().reset_index(drop=True)

y = df_tr["credit"].astype(int)
X = df_tr.drop(columns=["credit"]).copy()

# 제출/예측은 raw test를 사용해야 10000 보장
X_test = df_te_raw.copy()

# 컬럼 분리
num_cols = [c for c in X.columns if is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]
cat_features = [X.columns.get_loc(c) for c in cat_cols]

# 수치 결측 처리(훈련 중앙값 기준)
for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

# 범주 처리
for c in cat_cols:
    X[c] = X[c].astype(str).fillna("MISSING")
    X_test[c] = X_test[c].astype(str).fillna("MISSING")

n_classes = y.nunique()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_pred = np.zeros((len(X_test), n_classes))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    tr_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_val, y_val, cat_features=cat_features)
    te_pool  = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=6000,
        learning_rate=0.03,
        #class_weights=class_weights,
        depth=8,
        l2_leaf_reg=6,
        random_seed=42,
        verbose=200,
        early_stopping_rounds=300
    )

    model.fit(tr_pool, eval_set=val_pool, use_best_model=True)
    test_pred += model.predict_proba(te_pool) / skf.n_splits

print("✅ test_pred shape:", test_pred.shape)  # (10000, 3) 이어야 정상

0:	learn: 1.0817741	test: 1.0817029	best: 1.0817029 (0)	total: 54.7ms	remaining: 5m 27s
200:	learn: 0.8014241	test: 0.8059094	best: 0.8059094 (200)	total: 9.51s	remaining: 4m 34s
400:	learn: 0.7779628	test: 0.7987983	best: 0.7987983 (400)	total: 21.4s	remaining: 4m 58s
600:	learn: 0.7471641	test: 0.7904216	best: 0.7904216 (600)	total: 33.4s	remaining: 4m 59s
800:	learn: 0.7216593	test: 0.7850692	best: 0.7850595 (798)	total: 45.5s	remaining: 4m 55s
1000:	learn: 0.6964702	test: 0.7802175	best: 0.7802175 (1000)	total: 57.6s	remaining: 4m 47s
1200:	learn: 0.6745638	test: 0.7774209	best: 0.7774076 (1194)	total: 1m 9s	remaining: 4m 38s
1400:	learn: 0.6535552	test: 0.7737603	best: 0.7737588 (1399)	total: 1m 21s	remaining: 4m 28s
1600:	learn: 0.6323589	test: 0.7714738	best: 0.7714114 (1593)	total: 1m 33s	remaining: 4m 17s
1800:	learn: 0.6127139	test: 0.7698195	best: 0.7697742 (1795)	total: 1m 45s	remaining: 4m 6s
2000:	learn: 0.5935847	test: 0.7677165	best: 0.7676723 (1999)	total: 1m 57s	remai

In [13]:
print("df_te_raw:", df_te_raw.shape)
print("X_test shape:", X_test.shape)        # 네가 모델에 넣는 test
print("test_pred shape:", test_pred.shape)

df_te_raw: (10000, 19)
X_test shape: (10000, 19)
test_pred shape: (10000, 3)


In [14]:
# fold loop 안에서 model 생성 시 추가
# class_weights는 (0,1,2) 순서
class_counts = y.value_counts().sort_index().values
class_weights = (class_counts.sum() / (len(class_counts) * class_counts)).tolist()
print("class_counts:", class_counts, "class_weights:", class_weights)

class_counts: [ 3060  5931 15820] class_weights: [2.702723311546841, 1.3944247737874444, 0.5227770754319427]


In [15]:
start_idx = len(df_tr_raw)  # 26457
submit_index = np.arange(start_idx, start_idx + len(df_te_raw))

sub = pd.DataFrame({
    "index": submit_index,
    "0": test_pred[:, 0],
    "1": test_pred[:, 1],
    "2": test_pred[:, 2],
})

sub.to_csv("../../data/submission_cat_v2.csv", index=False)
print("saved: ../../data/submission_cat_v2.csv")
print(sub.head())

saved: ../../data/submission_cat_v2.csv
   index         0         1         2
0  26457  0.086464  0.093845  0.819691
1  26458  0.164124  0.143240  0.692636
2  26459  0.126142  0.150852  0.723006
3  26460  0.130821  0.133361  0.735818
4  26461  0.091555  0.184849  0.723597


In [16]:
import numpy as np
import pandas as pd

TRAIN_PATH = "../../data/train_processed.csv"
TEST_PATH  = "../../data/test_processed.csv"

# 제출 규격은 raw 기준이어야 함
df_tr_raw = pd.read_csv(TRAIN_PATH)
df_te_raw = pd.read_csv(TEST_PATH)

# 1) 기본 체크 (여기서 하나라도 틀리면 제출 에러 가능성 큼)
print("raw train rows:", len(df_tr_raw))  # 26457 기대
print("raw test rows :", len(df_te_raw))  # 10000 기대
print("test_pred shape:", getattr(test_pred, "shape", None))  # (10000, 3) 기대

# 2) test_pred 강제 정리 (NaN/Inf/음수/합 1 보정)
pred = np.array(test_pred, dtype=float)

# 행 수/클래스 수 맞추기
assert pred.shape[0] == len(df_te_raw), "test_pred 행 수가 raw test(10000)과 다름"
assert pred.shape[1] == 3, "클래스 수가 3이 아님"

pred = np.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
pred = np.clip(pred, 0.0, 1.0)

row_sum = pred.sum(axis=1, keepdims=True)
# 합이 0인 행 방지
row_sum[row_sum == 0] = 1.0
pred = pred / row_sum

# 3) index 생성 (반드시 raw train 길이 기준)
start_idx = len(df_tr_raw)  # 26457
submit_index = np.arange(start_idx, start_idx + len(df_te_raw))

# 4) 제출 df 생성 (컬럼명/순서 고정)
sub = pd.DataFrame({
    "index": submit_index.astype(int),
    "0": pred[:, 0].astype(float),
    "1": pred[:, 1].astype(float),
    "2": pred[:, 2].astype(float),
})

# 5) 최종 검증
assert list(sub.columns) == ["index", "0", "1", "2"]
assert len(sub) == 10000
assert sub["index"].iloc[0] == 26457 and sub["index"].iloc[-1] == 36456
assert np.isfinite(sub[["0","1","2"]].to_numpy()).all()
assert (sub[["0","1","2"]].sum(axis=1).between(0.999, 1.001)).all()

out_path = "../../data/submission_cat_v2.csv"
sub.to_csv(out_path, index=False)
print("✅ saved:", out_path)
print(sub.head())

raw train rows: 26457
raw test rows : 10000
test_pred shape: (10000, 3)
✅ saved: ../../data/submission_cat_v2.csv
   index         0         1         2
0  26457  0.086464  0.093845  0.819691
1  26458  0.164124  0.143240  0.692636
2  26459  0.126142  0.150852  0.723006
3  26460  0.130821  0.133361  0.735818
4  26461  0.091555  0.184849  0.723597
